# Exercise 2 — Image Classification

**CAS Deep Learning — Computer Vision**

## Learning objectives
After this exercise you should be able to:
- Set up a complete supervised image classification pipeline in PyTorch
- Use transfer learning with a pretrained EfficientNet from `timm`
- Write a training loop that monitors training and validation metrics
- Interpret loss/accuracy curves and identify overfitting
- Inspect model predictions and reason about errors

**Fill in code where you see `# YOUR CODE HERE`.**

## Setup Colab / Local Paths

The following setup sets the default data path: Make sure to check and adapt `DATA_PATH`.

If working in Colab: **Save the notebook to your personal Google Drive to persist changes**.

Packages not in the Colab environment will be installed as well.

In [ ]:
import sys
from pathlib import Path

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")


if IN_COLAB:
    # Mirrors requirements.txt — keep them in sync.
    print("Installing additional packages to Colab Environment")
    !pip install -q timm torchinfo gdown huggingface_hub

    print("Mounting Drive")
    from google.colab import drive

    ROOT_PATH = Path("/content/drive")
    drive.mount(str(ROOT_PATH))
    DATA_PATH = ROOT_PATH.joinpath("MyDrive/cas-dl-compvis")
else:
    import pyrootutils

    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")

print(f"Creating {DATA_PATH} if it does not exist")
DATA_PATH.mkdir(parents=True, exist_ok=True)

## Imports

In [ ]:
import time

import matplotlib.pyplot as plt
import timm
import torch
import torch.nn as nn
import torchinfo
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm.notebook import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device} | Colab: {IN_COLAB}")

## Load the Dataset

We use the same Serengeti camera trap dataset from Exercise 1. The dataset is organized in ImageFolder layout with train/val/test splits.

If you prefer to use the CCT-20 dataset instead, uncomment the alternative block below.

In [ ]:
def ensure_dataset(
    data_path,
    archive_name,
    extracted_subdir,
    *,
    gdrive_id=None,
    hf_repo=None,
):
    """Check-then-download a dataset archive. Returns the extracted directory.

    Tries Google Drive first (gdown), falls back to Hugging Face Hub. Supports
    .tar.gz and .zip. Idempotent — skips if the extracted directory exists.
    """
    data_path = Path(data_path)
    dataset_dir = data_path / extracted_subdir
    if dataset_dir.exists():
        print(f"Dataset already present: {dataset_dir}")
        return dataset_dir

    data_path.mkdir(parents=True, exist_ok=True)
    archive_path = data_path / archive_name

    if not archive_path.exists():
        errors = []
        if gdrive_id is not None:
            try:
                import gdown

                url = f"https://drive.google.com/uc?id={gdrive_id}"
                print(f"Downloading {archive_name} from Google Drive ...")
                gdown.download(url, str(archive_path), quiet=False, fuzzy=True)
                if not archive_path.exists():
                    raise RuntimeError("gdown reported success but file is missing")
            except Exception as exc:
                errors.append(f"gdown: {exc}")
                archive_path.unlink(missing_ok=True)

        if not archive_path.exists() and hf_repo is not None:
            try:
                from huggingface_hub import hf_hub_download

                print(f"Downloading {archive_name} from Hugging Face ({hf_repo}) ...")
                cached = hf_hub_download(hf_repo, archive_name, repo_type="dataset")
                if Path(cached).resolve() != archive_path.resolve():
                    archive_path.write_bytes(Path(cached).read_bytes())
            except Exception as exc:
                errors.append(f"hf_hub_download: {exc}")

        if not archive_path.exists():
            sources = ", ".join(errors) if errors else "no sources configured"
            raise RuntimeError(f"Could not fetch {archive_name}. Tried: {sources}.")

    print(f"Extracting {archive_path.name} to {data_path} ...")
    if archive_name.endswith((".tar.gz", ".tgz")):
        import tarfile

        with tarfile.open(archive_path) as tar:
            tar.extractall(data_path)
    elif archive_name.endswith(".zip"):
        import zipfile

        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(data_path)
    else:
        raise ValueError(f"Unsupported archive format: {archive_name}")

    assert dataset_dir.exists(), (
        f"Extraction of {archive_path.name} did not produce {dataset_dir}."
    )
    print(f"Ready: {dataset_dir}")
    return dataset_dir


# Default: Serengeti balanced dataset (same as Exercise 1).
DATASET_PATH = ensure_dataset(
    DATA_PATH,
    archive_name="ser_balanced.tar.gz",
    extracted_subdir="ser/ser_balanced",
    gdrive_id="1iRlZue4-Udg_lA9RCYtg3zhctqbbPhX7",
    hf_repo="marco-willi/ser_balanced",
)

# --- Alternative: CCT-20 (8 balanced classes) or Kgalagadi (6, imbalanced) ---
# Uncomment one of the blocks below to use a different camera-trap dataset.
#
# DATASET_PATH = ensure_dataset(
#     DATA_PATH,
#     archive_name="cct20.tar.gz",
#     extracted_subdir="cct20",
#     gdrive_id="105DkEQcFhgWsQEzKh6p-u2QMMuUc2yt2",
#     hf_repo="marco-willi/camera-trap-cct20",
# )
#
# DATASET_PATH = ensure_dataset(
#     DATA_PATH,
#     archive_name="kgalagadi.tar.gz",
#     extracted_subdir="kgalagadi",
#     gdrive_id="129vX_GF4vUgwRlyLpx5BNPf6lI2n89Wp",
# )

print(f"Dataset path: {DATASET_PATH}")

In [ ]:
# Quick check: list classes and count
raw_train = ImageFolder(root=DATASET_PATH / "train")
NUM_CLASSES = len(raw_train.classes)
print(f"Classes ({NUM_CLASSES}): {raw_train.classes}")
print(f"Training images: {len(raw_train)}")

## Part 1 — Data Pipeline

A solid data pipeline is the foundation of any classification project. We define separate transform pipelines for training and validation:

- **Training transforms** include random augmentations (horizontal flip, color jitter) to improve generalization.
- **Validation transforms** are deterministic — we want reproducible evaluation.

Both pipelines produce 300×300 crops and apply ImageNet normalization — this matches the native input size of `efficientnet_b3` (our default backbone in Part 2).

**Matching the pretrained model.** A pretrained backbone expects inputs that look like the ones it saw during pretraining — same resolution, same normalization, and same resize interpolation. For `efficientnet_b3` in `timm`, the pretrained config is: input 300×300, crop pct 0.904 (resize to 332, then center-crop 300), ImageNet mean/std, and **bicubic** interpolation. (If you swap to `efficientnet_b0` as the "fast" alternative, its native input is 224×224 with resize to 256 — same normalization and interpolation.)

In practice, the robust way to obtain the config is to ask the model directly:

```python
data_config = timm.data.resolve_model_data_config(model)
val_transforms = timm.data.create_transform(**data_config, is_training=False)
```

We write the transforms explicitly here so each step is visible.

Define `train_transforms` and `val_transforms`:

- **`train_transforms`**: `Resize(332, bicubic)` → `RandomCrop(300)` → `RandomHorizontalFlip` → `ColorJitter(brightness=0.2, contrast=0.2)` → `ToTensor` → `Normalize`
- **`val_transforms`**: `Resize(332, bicubic)` → `CenterCrop(300)` → `ToTensor` → `Normalize`

Use ImageNet statistics: `mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`. Use **bicubic** interpolation in `Resize` to match how `efficientnet_b3` was pretrained.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


Create `train_dataset` and `val_dataset` using `ImageFolder` with the transforms defined above. Then wrap each in a `DataLoader` with `batch_size=32`. Shuffle the training loader.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
# Verify: load one batch and check shape
images, labels = next(iter(train_loader))
assert images.shape == (BATCH_SIZE, 3, 300, 300), f"Got {images.shape}"
assert labels.shape == (BATCH_SIZE,), f"Got {labels.shape}"
print(f"Batch shape: {images.shape}")
print("OK")

**Question:** Why do we use different transforms for training and validation?

<details>
<summary>Click to reveal answer</summary>

**Training transforms** include random augmentations (flips, crops, color jitter) that expose the model to varied views of each image. This acts as a regularizer and helps the model generalize to unseen data.

**Validation transforms** are deterministic (center crop, no randomness) so that evaluation is reproducible. We want to measure model performance on a consistent, unaugmented view of the data.

Mixing augmentation into validation would make metrics noisy and unreliable.

</details>

### What goes into the model?

Before moving on, let's visualize what the model actually sees. We apply the training transforms to a few images and display both the original and the transformed version side by side. This is a useful sanity check — the augmented images should still look plausible.

In [ ]:
# Show 4 images: original (top) vs. augmented+normalized (bottom, denormalized for display)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

mean = torch.tensor(imagenet_mean).view(3, 1, 1)
std = torch.tensor(imagenet_std).view(3, 1, 1)

raw_dataset = ImageFolder(root=DATASET_PATH / "train")  # no transforms

for col in range(4):
    img_pil, label = raw_dataset[col]
    class_name = raw_dataset.classes[label]

    # Original
    _ = axes[0, col].imshow(img_pil)
    _ = axes[0, col].set_title(f"{class_name}", fontsize=10)

    # Transformed (denormalized for display)
    img_t = train_transforms(img_pil)
    img_display = torch.clamp(img_t * std + mean, 0, 1)
    _ = axes[1, col].imshow(img_display.permute(1, 2, 0).numpy())
    _ = axes[1, col].set_title("Transformed", fontsize=10)

# Hide ticks and spines but keep row ylabels visible
for ax in axes.flat:
    _ = ax.set_xticks([])
    _ = ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

_ = axes[0, 0].set_ylabel("Original", fontsize=12)
_ = axes[1, 0].set_ylabel("Augmented", fontsize=12)
_ = plt.suptitle(
    "What goes into the model? (augmented images, denormalized for display)",
    fontsize=13,
)
_ = plt.tight_layout()
plt.show()

**Question:** Do the augmented images still look plausible? Could any of the augmentations introduce artifacts that might hurt training?

<details>
<summary>Click to reveal answer</summary>

The augmented images should still look like realistic camera trap photos — just cropped differently, sometimes flipped, and with slight color variation. This is the point of data augmentation: create realistic variations that help the model generalize.

Augmentations that could hurt: too aggressive color jitter can make images look unnatural, and random crops might cut out the animal entirely. For camera trap images where the animal can be small, `RandomResizedCrop` with a very small scale could be problematic. Our pipeline uses `Resize(256)` + `RandomCrop(224)`, which is mild — only a small border region is cropped.

</details>

## Part 2 — Model Setup

Training a CNN from scratch requires large datasets and significant compute. **Transfer learning** sidesteps this by starting from a model pretrained on ImageNet (1.2 million images, 1000 classes). The pretrained feature extractor already understands edges, textures, and shapes — we only need to adapt the final classification head to our specific task.

We use the `timm` library to load a pretrained **EfficientNet-B3** and replace the classification head. B3 is a good default: ~12M parameters, 300×300 input, and noticeably stronger features than the smallest variant.

**Alternatives** (just change the model name):
- `efficientnet_b0` — ~5M params, 224×224 input. Use this when you want **very fast training** (e.g. CPU-only Colab) and are willing to trade some accuracy. Remember to switch the Part 1 transforms to `Resize(256)` / `CenterCrop(224)`.
- `resnet50` — the classic 25M-param CNN baseline; 224×224 input.

Load a pretrained `efficientnet_b3` from `timm` with the correct number of output classes. Then freeze all parameters except the classification head.

Hints:
- `timm.create_model('efficientnet_b3', pretrained=True, num_classes=NUM_CLASSES)`
- Freeze: set `param.requires_grad = False` for all parameters
- Unfreeze: set `param.requires_grad = True` for the classifier parameters
- Use `model.get_classifier()` to access the classification head
- For very fast training, replace `'efficientnet_b3'` with `'efficientnet_b0'` (and switch the transforms back to 256/224). For a ResNet baseline, use `'resnet50'`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
# Verify: count trainable vs total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Frozen parameters:    {frozen_params:>12,}")
print(f"Trainable fraction:   {trainable_params / total_params:.2%}")

assert trainable_params < total_params, "Some parameters should be frozen"
assert trainable_params > 0, "Classification head should be trainable"
print("OK")

Let's inspect the model architecture. The `torchinfo` summary shows the layer structure, output shapes, and parameter counts.

In [ ]:
torchinfo.summary(model, input_size=(1, 3, 300, 300), device=device)

**Question:** Why do we freeze the pretrained layers and only train the classification head?

<details>
<summary>Click to reveal answer</summary>

1. **Speed:** Training only the head is much faster — we update <1% of all parameters.
2. **Data efficiency:** With a small dataset (~1,600 images), training all 25M parameters would overfit quickly. The pretrained features are already strong enough.
3. **Stability:** The pretrained feature extractor has been trained on millions of images. Fine-tuning all layers with a high learning rate could destroy those useful representations ("catastrophic forgetting").

This strategy is sometimes called **feature extraction** — we use the pretrained backbone purely as a fixed feature extractor and only learn a new classification head.

</details>

**Question:** Could we skip running the backbone every epoch and just **pre-compute embeddings once**, then train only a classifier on those frozen features?

<details>
<summary>Click to reveal answer</summary>

In principle yes — if the backbone is frozen, its output for a given image never changes. Pre-computing embeddings once and training a small head on the cached vectors would be dramatically faster.

**The catch: no data augmentation.** Augmentations (random crops, flips, color jitter) happen *before* the image enters the backbone, so each epoch the backbone sees a different transformed view of the same image. Caching embeddings freezes a single view per image and throws that regularization signal away — the model sees less variety and tends to overfit faster on small datasets.

**When pre-computed embeddings do make sense:**
- Very large datasets where augmentation matters less than throughput
- Retrieval / k-NN use-cases that do not involve training at all (see Exercise 3)
- Quick linear-probe baselines, where you accept the loss of augmentation for speed

</details>

## Part 3 — Training Loop

We now implement a standard PyTorch training loop. The loop iterates over the training data, computes the loss, backpropagates gradients, and updates the trainable parameters. After each epoch, we evaluate on the validation set to monitor generalization.

We use:
- **CrossEntropyLoss** — the standard loss for multi-class classification
- **Adam optimizer** — a good default optimizer with adaptive learning rates

Define the loss function (`nn.CrossEntropyLoss`) and the optimizer (`torch.optim.Adam`). Only pass the **trainable** parameters to the optimizer, with `lr=1e-3`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
assert isinstance(criterion, nn.CrossEntropyLoss)
print(f"Optimizer: {optimizer.__class__.__name__}, lr={optimizer.defaults['lr']}")
print("OK")

Now implement the training loop. We provide helper functions for one training epoch and one evaluation pass. Complete the training loop that runs for `NUM_EPOCHS=5` and stores the metrics history.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: str = "cpu",
) -> tuple[float, float]:
    """Train for one epoch. Returns (avg_loss, accuracy)."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(data_loader, desc="Train", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


@torch.no_grad()
def evaluate(
    model: nn.Module,
    data_loader: DataLoader,
    criterion: nn.Module,
    device: str = "cpu",
) -> tuple[float, float]:
    """Evaluate on a dataset. Returns (avg_loss, accuracy)."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(data_loader, desc="Val", leave=False):
        images, labels = images.to(device), labels.to(device)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

**Question:** What loss value do you expect at initialization (before any training) for a model with 8 classes?

<details>
<summary>Click to reveal answer</summary>

With 8 classes and random predictions, the expected cross-entropy loss is:

$$-\log\left(\frac{1}{8}\right) = \log(8) \approx 2.08$$

If the initial loss is close to this value, it confirms the model is predicting roughly uniformly at the start. A loss significantly higher than this could indicate a numerical issue.

</details>

Write a training loop for `NUM_EPOCHS = 5` that:
1. Calls `train_one_epoch` on the training data
2. Calls `evaluate` on the validation data
3. Prints the epoch metrics
4. Stores all metrics in a `history` dictionary with keys: `"train_loss"`, `"train_acc"`, `"val_loss"`, `"val_acc"` (each a list of values per epoch)

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
# Verify: history should have 5 entries per metric
for key in ["train_loss", "train_acc", "val_loss", "val_acc"]:
    assert key in history, f"Missing key: {key}"
    assert len(history[key]) == NUM_EPOCHS, (
        f"{key} should have {NUM_EPOCHS} entries, got {len(history[key])}"
    )
print(f"Final val accuracy: {history['val_acc'][-1]:.3f}")
print("OK")

## Part 4 — Visualize Training Curves

Plotting the training history is essential for diagnosing model behavior. The gap between training and validation metrics tells us whether the model is underfitting, overfitting, or generalizing well.

Plot the training and validation **loss** and **accuracy** curves side by side. Use `history` from Part 3.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


**Question:** Does the model overfit? How can you tell from the curves? Would you train for more epochs?

<details>
<summary>Click to reveal answer</summary>

**Signs of overfitting:**
- Training loss keeps decreasing while validation loss starts increasing (or plateaus)
- A growing gap between training accuracy and validation accuracy

**Signs of good generalization:**
- Training and validation curves move together and both improve
- The gap between them remains small

With only 5 epochs of training the classification head, overfitting is unlikely. The frozen backbone acts as a strong regularizer. If you see signs of overfitting, common remedies include: more data augmentation, weight decay, early stopping, or reducing the learning rate.

Likely, the model can be trained even longer, paricularly if training loss is still decreasing.

A good strategy is to use early-stopping to dynamically stop training once validation loss plateaus.

</details>

**Question:** It is possible the validation loss is smaller than training loss (see learning cuve) in the first epochs. If that is the case, what could be the reason?

<details>
<summary>Click to reveal answer</summary>

To calculate training loss, the loss of each batch during model training is recorded and averaged at the **end** of each epoch. Because the model improves during model training, the measured **average** loss on the training set might be higher than for the validation set which is evaluated at the **end** of each epoch. This turns around as overfitting starts to dominate.

</details>

## Part 5 - Inspect Predictions

Aggregate metrics (accuracy, loss) only tell part of the story. Inspecting individual predictions helps identify systematic errors — for example, does the model confuse specific animal species?

Write a function that takes a batch of images and labels, runs the model, and returns the predicted class indices and confidence scores (softmax probabilities).

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
# Verify: predict on one batch
test_images, test_labels = next(iter(val_loader))
preds, confs = predict_batch(model, test_images, device)
assert preds.shape == test_labels.shape
assert confs.shape == test_labels.shape
assert (confs >= 0).all() and (confs <= 1).all(), "Confidences should be in [0, 1]"
print("OK")

Display 8 validation images with their predicted label, confidence, and true label. Highlight incorrect predictions in red.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


Finally, let's compute the per-class accuracy to see if the model struggles with specific animal categories.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


A per-class accuracy table shows how well the model performs on each class in isolation, but it does not reveal **which classes get confused with each other**. A confusion matrix fills that gap: rows are the true classes, columns the predicted classes, and each cell counts how often that (true, predicted) combination occurs. The diagonal is correct; off-diagonal entries expose systematic mix-ups.

We use scikit-learn's `confusion_matrix` and `ConfusionMatrixDisplay` to compute and plot it.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


**Question:** Which classes does the model find most difficult? Why might that be?

<details>
<summary>Click to reveal answer</summary>

Common confusion patterns in camera trap datasets:
- **"empty" vs. animals** — some "empty" images have faint animal traces, and some animal images show very little of the animal.
- **Visually similar species** (e.g., coyote vs. cat) — camera trap images are often low-resolution, nighttime, or partially occluded.
- **Small or distant animals** — when the animal occupies only a few pixels, the model relies on background context rather than the animal itself.

Inspecting per-class metrics is critical. A model with 90% overall accuracy might still fail completely on one class.

</details>

## Summary

In this exercise you built a complete image classification pipeline:

1. **Data pipeline** — separate transforms for training (with augmentation) and validation (deterministic), sized to match the pretrained backbone
2. **Transfer learning** — pretrained EfficientNet-B3 backbone with a new classification head (swap to `efficientnet_b0` for very fast training, or `resnet50` as a CNN baseline)
3. **Training loop** — standard PyTorch loop with training and validation metrics per epoch
4. **Evaluation** — loss/accuracy curves to diagnose overfitting, per-class analysis, visual inspection

**Key takeaways:**
- Transfer learning with a frozen backbone is fast, data-efficient, and often sufficient for small datasets
- The preprocessing pipeline must match the backbone's pretrained config (resolution, crop pct, interpolation, normalization)
- Always monitor validation metrics alongside training metrics — training accuracy alone is misleading
- Inspect individual predictions and per-class performance to understand where the model fails